In [1]:
import glob
import os
import zipfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ==========================================
# 1. Unzip and Load Dataset
# ==========================================
zip_path = "/content/archive(4).zip"
extract_dir = "/content/churn_dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
  zip_ref.extractall(extract_dir)

# Automatically locate the CSV file inside the unzipped contents
csv_files = glob.glob(f"{extract_dir}/**/*.csv", recursive=True)
if not csv_files:
  raise FileNotFoundError("No CSV file found inside the extracted archive.")

df = pd.read_csv(csv_files[0])
print(f"Loaded: {os.path.basename(csv_files[0])} | Shape: {df.shape}")

# ==========================================
# 2. Identify Target & Clean Features
# ==========================================
# Identify common churn column naming conventions
target_candidates = ["Churn", "churn", "Exited", "target", "churn_value"]
target_col = next((col for col in target_candidates if col in df.columns), None)

if not target_col:
  raise KeyError(
      f"Target column not found. Available columns: {list(df.columns)}"
  )

# Convert string target (e.g., 'Yes'/'No') or boolean to integer (1/0)
if df[target_col].dtype == "object":
  df[target_col] = (
      df[target_col].str.strip().map({"Yes": 1, "No": 0, "True": 1, "False": 0})
  )
elif df[target_col].dtype == "bool":
  df[target_col] = df[target_col].astype(int)

# Drop missing target values
df = df.dropna(subset=[target_col])

# Drop common non-predictive identifier columns
id_cols = [
    "customerID",
    "CustomerID",
    "id",
    "ID",
    "RowNumber",
    "Surname",
    "Name",
]
df = df.drop(columns=[col for col in id_cols if col in df.columns])

# Fix common whitespace-as-null issue in numeric fields (e.g., 'TotalCharges')
for col in df.select_dtypes(include=["object"]).columns:
  converted = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
  # If converting reduces less than 5% of non-nulls to NaN, treat as numeric
  if (
      converted.notnull().sum() / len(df) > 0.95
      and converted.nunique() > df[col].nunique() * 0.5
  ):
    df[col] = converted

# Separate features (X) and label (y)
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# Identify numerical and categorical feature sets
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Fill missing numerical values with median and categorical with mode
for col in num_cols:
  X[col] = X[col].fillna(X[col].median())
for col in cat_cols:
  X[col] = X[col].fillna(X[col].mode()[0] if not X[col].mode().empty else "None")

# ==========================================
# 3. Train-Test Split & Preprocessing Pipeline
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ]
)

# ==========================================
# 4. Model Training & Comparison
# ==========================================
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42
    ),
}

results = []

for name, model in models.items():
  pipeline = Pipeline(
      steps=[("preprocessor", preprocessor), ("classifier", model)]
  )

  pipeline.fit(X_train, y_train)
  y_pred = pipeline.predict(X_test)
  y_prob = pipeline.predict_proba(X_test)[:, 1]

  acc = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  roc = roc_auc_score(y_test, y_prob)

  results.append({"Model": name, "Accuracy": acc, "F1-Score": f1, "ROC-AUC": roc})

  print(f"\n{'='*25} {name} {'='*25}")
  print(classification_report(y_test, y_pred, digits=4))

# ==========================================
# 5. Summary Table
# ==========================================
results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
print("\n--- Final Performance Comparison ---")
print(results_df.to_string(index=False))

Loaded: Churn_Modelling.csv | Shape: (10000, 14)

========================= Logistic Regression =========================
              precision    recall  f1-score   support

           0     0.9047    0.7150    0.7987      1593
           1     0.3873    0.7052    0.5000       407

    accuracy                         0.7130      2000
   macro avg     0.6460    0.7101    0.6494      2000
weighted avg     0.7994    0.7130    0.7379      2000


========================= Random Forest =========================
              precision    recall  f1-score   support

           0     0.9061    0.8964    0.9012      1593
           1     0.6108    0.6364    0.6233       407

    accuracy                         0.8435      2000
   macro avg     0.7585    0.7664    0.7623      2000
weighted avg     0.8460    0.8435    0.8447      2000


========================= Gradient Boosting =========================
              precision    recall  f1-score   support

           0     0.8788    0.96